In [1]:
import asyncio
import base64
from io import BytesIO, StringIO
from uuid import uuid4

import matplotlib.pyplot as plt
import pandas as pd
import uvicorn
from crewai import Agent, Crew
from crewai import Task as CrewTask
from crewai.process import Process
from crewai.tools import tool
from dotenv import load_dotenv

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentSkill,
    Part,
    Task,
)
from a2a.utils import completed_task, new_artifact
from a2a.utils.errors import InternalError, UnsupportedOperationError

load_dotenv()

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


True

In [2]:
# Plain dict to pass image data from the tool to the executor
image_cache: dict[str, dict] = {}


@tool('ChartGenerationTool')
def generate_chart_tool(prompt: str, session_id: str) -> str:
    """Generates a bar chart image from CSV input using matplotlib."""
    df = pd.read_csv(StringIO(prompt))
    df.columns = ['Category', 'Value']
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

    fig, ax = plt.subplots()
    ax.bar(df['Category'], df['Value'])
    ax.set_xlabel('Category')
    ax.set_ylabel('Value')
    ax.set_title('Bar Chart')

    buf = BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)

    image_id = uuid4().hex
    image_cache[f'{session_id}:{image_id}'] = {
        'bytes': base64.b64encode(buf.read()).decode('utf-8'),
        'mime_type': 'image/png',
        'name': 'chart.png',
    }
    return image_id


c_agent=Agent(
            role='Chart Creation Expert',
            goal='Generate a bar chart image based on structured CSV input.',
            backstory='You are a data visualization expert.',
            verbose=False,
            allow_delegation=False,
            tools=[generate_chart_tool],
        )
c_task=CrewTask(
            description=(
                "You are given a prompt: '{user_prompt}'.\n"
                "Reformat it into CSV with header 'Category,Value', "
                "then pass to 'ChartGenerationTool'.\n"
                "Use session ID: '{session_id}'."
            ),
            expected_output='The id of the generated chart image',
            agent=c_agent,  # set below
        )

In [3]:
chart_crew = Crew(
            agents=[c_agent],
            tasks=[c_task],
            process=Process.sequential,
            verbose=False,
        )

In [4]:
class ChartExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        query = context.get_user_input()
        try:
            result = chart_crew.kickoff({'user_prompt': query, 'session_id': context.context_id})
        except Exception as e:
            raise InternalError(f'Error invoking agent: {e}') from e

        image_data = image_cache.get(f'{context.context_id}:{result.raw}')
        if image_data:
            parts = [Part(
                raw=base64.b64decode(image_data['bytes']),
                media_type=image_data['mime_type'],
                filename=image_data['name'],
            )]
        else:
            parts = [Part(text='Failed to generate chart image.')]

        await event_queue.enqueue_event(
            completed_task(
                context.task_id,
                context.context_id,
                [new_artifact(parts, f'chart_{context.task_id}')],
                [context.message],
            )
        )

    async def cancel(self, request: RequestContext, event_queue: EventQueue) -> Task | None:
        raise UnsupportedOperationError()


agent_card = AgentCard(
    name='Chart Generator Agent',
    description='Generate charts from structured CSV-like data input.',
    supported_interfaces=[AgentInterface(url='http://localhost:10011/')],
    version='1.0.0',
    default_input_modes=['text', 'text/plain', 'image/png'],
    default_output_modes=['text', 'text/plain', 'image/png'],
    capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(
        id='chart_generator',
        name='Chart Generator',
        description='Generate a chart based on CSV-like data',
        tags=['generate image'],
        examples=['Generate a chart: Jan,1000 Feb,2000 Mar,1500'],
    )],
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=DefaultRequestHandler(
        agent_executor=ChartExecutor(),
        task_store=InMemoryTaskStore(),
    ),
).build()

In [5]:
config = uvicorn.Config(app=app, host='localhost', port=10011, log_level='info')
server = uvicorn.Server(config)
asyncio.ensure_future(server.serve())
print('Server running at http://localhost:10011')

Server running at http://localhost:10011


INFO:     Started server process [8455]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10011 (Press CTRL+C to quit)
